In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.append(str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/pqnhhh/Documents/GitHub/EL4TF


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch
from torch.utils.data import DataLoader, TensorDataset


import math
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_absolute_percentage_error

from loaders._load_vn30_multi_class_deep import preprocess, TARGETS
from models.lstm import LSTMClassifier

In [3]:
train_loader, valid_loader, test_loader, feature_scaler, classes = preprocess('VIC', verbose=True, val=0.1)

=== Preprocessing VIC ===
Train: (1108, 120), Val: (123, 120), Test: (286, 120)


In [4]:
model = LSTMClassifier(
    input_dim=120,
    hidden_dim=64,
    n_layers=2,
    dropout=0.2,
)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.8)
criterion = nn.CrossEntropyLoss()

In [5]:
best_val_loss = float('inf')
n_epochs = 50

for epoch in range(1, n_epochs + 1):
    # --- train ---
    model.train()
    train_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch, y_batch
        optimizer.zero_grad()
        preds = model(X_batch)
        loss  = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * X_batch.size(0)
    train_loss /= len(train_loader.dataset)

    # --- validation ---
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in valid_loader:
            X_batch, y_batch = X_batch, y_batch
            preds = model(X_batch)
            val_loss += criterion(preds, y_batch).item() * X_batch.size(0)
    val_loss /= len(valid_loader.dataset)

    scheduler.step()

    # --- checkpoint ---
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), f'checkpoints/lstm_VIC.pth')

    if epoch % 10 == 0 or epoch == n_epochs:
        print(f"Epoch {epoch:3d}/{n_epochs}: "
              f"Train Loss = {train_loss:.6f}, "
              f"Valid Loss = {val_loss:.6f}, "
              f"Best Val Loss = {best_val_loss:.6f}, "
			  f"LR = {optimizer.param_groups[0]['lr']:.6f}")

Epoch  10/50: Train Loss = 1.435495, Valid Loss = 1.594394, Best Val Loss = 1.562000, LR = 0.000800
Epoch  20/50: Train Loss = 1.435844, Valid Loss = 1.594519, Best Val Loss = 1.562000, LR = 0.000640
Epoch  30/50: Train Loss = 1.435021, Valid Loss = 1.592041, Best Val Loss = 1.562000, LR = 0.000512
Epoch  40/50: Train Loss = 1.432114, Valid Loss = 1.597949, Best Val Loss = 1.562000, LR = 0.000410
Epoch  50/50: Train Loss = 1.432431, Valid Loss = 1.596563, Best Val Loss = 1.562000, LR = 0.000328


In [6]:
import numpy as np
import torch
from sklearn.metrics import accuracy_score, f1_score, classification_report
from loaders._load_vn30_meta import VN30

# Giả sử eval_dict và các hàm/biến khác đã được định nghĩa
eval_dict = {}
for symbol in VN30:
    eval_dict[symbol] = []

def eval(symbol):
    # Giả sử hàm preprocess của bạn đã đúng cho bài toán classification
    # và model được lưu với tên lstm_{symbol}.pth
    train_loader, valid_loader, test_loader, feature_scaler, classes = preprocess(symbol, verbose=False, val=0.1)
    
    # Tải mô hình đã huấn luyện
    # Sửa tên file cho đúng với mô hình classification của bạn
    model.load_state_dict(torch.load(f'checkpoints/lstm_{symbol}.pth', map_location='cpu'))
    model.eval()

    # Thu thập dự đoán và nhãn thật
    all_preds   = []
    all_targets = []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch
            
            # 1. LẤY NHÃN DỰ ĐOÁN (thay vì logits)
            preds_logits = model(X_batch)
            # Dùng argmax để tìm index của lớp có logit cao nhất
            predicted_labels = torch.argmax(preds_logits, dim=1).cpu().numpy()
            
            all_preds.append(predicted_labels)
            all_targets.append(y_batch.numpy())

    # Nối kết quả từ các batch lại thành một mảng 1D
    all_preds   = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)

    # 2. BỎ BƯỚC INVERSE_TRANSFORM
    # Vì nhãn lớp là các chỉ số, không phải giá trị liên tục

    # 3. SỬ DỤNG CÁC ĐỘ ĐO PHÂN LOẠI
    accuracy = accuracy_score(all_targets, all_preds) * 100
    f1 = f1_score(all_targets, all_preds, average='weighted') # 'weighted' f1-score tốt cho dữ liệu mất cân bằng

    # In báo cáo chi tiết cho từng lớp
    print(f"--- Evaluation for {symbol} ---")
    print(classification_report(all_targets, all_preds, target_names=classes, zero_division=0))

    # Lưu kết quả
    eval_dict[symbol].append((accuracy, f1))
    print(f"Overall Test Accuracy: {accuracy:.2f}%")
    print(f"Overall Test Weighted F1-Score: {f1:.4f}")
    
# Ví dụ cách gọi hàm
# for symbol in VN30:
#     eval_classification(symbol)

In [7]:
eval('VIC')  # Thay 'VIC' bằng mã cổ phiếu bạn muốn đánh giá

--- Evaluation for VIC ---
              precision    recall  f1-score   support

 strong_down       0.00      0.00      0.00        17
   weak_down       0.00      0.00      0.00        51
    sideways       0.51      1.00      0.67       145
     weak_up       0.00      0.00      0.00        47
   strong_up       0.00      0.00      0.00        26

    accuracy                           0.51       286
   macro avg       0.10      0.20      0.13       286
weighted avg       0.26      0.51      0.34       286

Overall Test Accuracy: 50.70%
Overall Test Weighted F1-Score: 0.3411
